# Phase Boundary Cloud Notebook

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, trains the phase-boundary DEC and IDEC workflows from `Phase Boundary Testing/src/train_phase_boundary_models.py`, and can optionally push saved results back to GitHub.

Sensitive values such as GitHub token, commit username, and commit email are requested at runtime or read from environment variables. They are not hardcoded in the notebook.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install tensorflow scikit-learn pandas matplotlib h5py hyperspy ncempy tqdm
else:
    print('Colab dependency install cell skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/CMU-vPCF-Project.git'
REPO_DIR = '/content/CMU-vPCF-Project'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_WEIGHT_FILES_TO_GITHUB = False
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/CMU_vPCF_Project/PhaseBoundaryOutputs'
GIT_RESULTS_SUBDIR = 'Phase Boundary Testing/Results/Colab_Runs'
GIT_BRANCH = 'main'
RUN_TAG = 'phase_boundary_full_train'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

phase_boundary_dir = repo_root / 'Phase Boundary Testing'
module_dir = phase_boundary_dir / 'src'
data_dir = phase_boundary_dir / 'Data'
notebook_path = phase_boundary_dir / 'Phase_Boundary_Training_Colab.ipynb'

if not (module_dir / 'train_phase_boundary_models.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/phase_boundary_outputs') if IN_COLAB else phase_boundary_dir / 'Results' / 'Local_Notebook_Runs'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_TAG}_{RUN_STAMP}'
output_dir.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

H5_FILE = data_dir / 'SyntheticModel_HfO_80pm_vPCFs_65.h5'
DM3_FILE = data_dir / 'SyntheticModel_HfO_80pm_gaussian_HAADF.dm3'

print(f'Repo root: {repo_root}')
print(f'Phase boundary dir: {phase_boundary_dir}')
print(f'Data dir: {data_dir}')
print(f'H5 file: {H5_FILE}')
print(f'DM3 file: {DM3_FILE}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Save outputs to Drive: {SAVE_OUTPUTS_TO_DRIVE}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'Push weight files to GitHub: {PUSH_WEIGHT_FILES_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')

In [ ]:
import json
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from train_phase_boundary_models import run_pipeline


def copy_run_tree(source_dir, dest_dir, include_weights=False):
    source_dir = Path(source_dir)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)

    for path in source_dir.rglob('*'):
        if path.is_dir():
            continue
        if not include_weights and path.suffixes[-2:] == ['.weights', '.h5']:
            continue
        relative_path = path.relative_to(source_dir)
        destination = dest_dir / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)


def git(args, check=True, capture_output=False, text=True):
    return subprocess.run(
        ['git', '-C', str(repo_root), *args],
        check=check,
        capture_output=capture_output,
        text=text,
    )

In [ ]:
TRAINING_KWARGS = {
    'model': 'both',
    'n_clusters': 3,
    'feature_method': 'flatten',
    'normalize': 'minmax',
    'downsample_factor': None,
    'max_frames': None,
    'hidden_dims': [500, 500, 2000],
    'pretrain_epochs': 50,
    'maxiter': 8000,
    'update_interval': 140,
    'batch_size': 256,
    'gamma': 0.1,
    'verbose': True,
}

RUN_CONFIGS = [
    {
        'name': 'h5_only',
        'h5_file': str(H5_FILE),
        'dm3_file': None,
    },
    {
        'name': 'dm3_only',
        'h5_file': None,
        'dm3_file': str(DM3_FILE),
    },
    {
        'name': 'combined',
        'h5_file': str(H5_FILE),
        'dm3_file': str(DM3_FILE),
    },
]

print(f"Training mode: {TRAINING_KWARGS['model']}")
print(f"Feature method: {TRAINING_KWARGS['feature_method']}")
print(f"Normalize: {TRAINING_KWARGS['normalize']}")
print(f"Pretrain epochs: {TRAINING_KWARGS['pretrain_epochs']}")
print(f"Max iterations: {TRAINING_KWARGS['maxiter']}")
print(f"Configured runs: {[config['name'] for config in RUN_CONFIGS]}")

In [ ]:
file_rows = []
for data_path in (H5_FILE, DM3_FILE):
    file_rows.append(
        {
            'path': str(data_path),
            'exists': data_path.exists(),
            'size_mb': round(data_path.stat().st_size / (1024 * 1024), 2) if data_path.exists() else None,
        }
    )

data_frame = pd.DataFrame(file_rows)
display(data_frame)

if not data_frame['exists'].all():
    raise FileNotFoundError('One or more required phase-boundary input files are missing.')

In [ ]:
run_summaries = []
saved_run_dirs = []

for config in RUN_CONFIGS:
    config_output_dir = output_dir / config['name']
    print('\n' + '=' * 80)
    print(f"Starting run: {config['name']}")
    print('=' * 80)

    results = run_pipeline(
        h5_file=config['h5_file'],
        dm3_file=config['dm3_file'],
        output_dir=str(config_output_dir),
        **TRAINING_KWARGS,
    )

    summary_path = Path(results['summary_path'])
    with open(summary_path, 'r', encoding='utf-8') as handle:
        summary = json.load(handle)

    summary['run_name'] = config['name']
    summary['output_dir'] = str(config_output_dir)
    run_summaries.append(summary)
    saved_run_dirs.append(config_output_dir)

saved_dir = output_dir
print(f'Training finished for all configured runs. Saved outputs to {saved_dir}')

In [ ]:
summary_rows = []

for summary in run_summaries:
    for model_name, metrics in summary.get('models', {}).items():
        summary_rows.append(
            {
                'run_name': summary['run_name'],
                'model': model_name,
                'n_samples': metrics.get('n_samples'),
                'n_clusters': metrics.get('n_clusters'),
                'silhouette_score': metrics.get('silhouette_score'),
                'davies_bouldin_score': metrics.get('davies_bouldin_score'),
                'calinski_harabasz_score': metrics.get('calinski_harabasz_score'),
                'cluster_size_min': metrics.get('cluster_size_min'),
                'cluster_size_max': metrics.get('cluster_size_max'),
                'output_dir': summary['output_dir'],
            }
        )

metrics_frame = pd.DataFrame(summary_rows)
manifest_path = saved_dir / 'phase_boundary_run_manifest.json'
metrics_csv_path = saved_dir / 'phase_boundary_metrics_summary.csv'

with open(manifest_path, 'w', encoding='utf-8') as handle:
    json.dump(run_summaries, handle, indent=2)
metrics_frame.to_csv(metrics_csv_path, index=False)

display(metrics_frame)

saved_files = sorted(str(path.relative_to(saved_dir)) for path in saved_dir.rglob('*') if path.is_file())
print(f'Saved artifacts to {saved_dir}')
print(f'Latest run pointer: {output_root / "latest_run.txt"}')
print('Saved files:')
for name in saved_files:
    print(f' - {name}')

In [ ]:
if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')
    if not GIT_COMMIT_USERNAME or not GIT_COMMIT_EMAIL:
        raise ValueError('Set commit username and email before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / saved_dir.name
    git_run_dir.mkdir(parents=True, exist_ok=True)
    copy_run_tree(saved_dir, git_run_dir, include_weights=PUSH_WEIGHT_FILES_TO_GITHUB)
    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    git(['config', 'user.name', GIT_COMMIT_USERNAME])
    git(['config', 'user.email', GIT_COMMIT_EMAIL])

    remote_url = git(['remote', 'get-url', 'origin'], capture_output=True).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    git(['remote', 'set-url', 'origin', auth_url])

    try:
        git([
            'add',
            str(git_run_dir),
            str(git_output_root / 'latest_run.txt'),
            str(notebook_path),
        ])
        diff_result = git(['diff', '--cached', '--quiet'], check=False, capture_output=False, text=False)
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add phase boundary cloud results for {saved_dir.name}'
            git(['commit', '-m', commit_message])
            git(['push', 'origin', GIT_BRANCH])
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        git(['remote', 'set-url', 'origin', remote_url])
else:
    print('GitHub push disabled. Results remain in the current run directory and any configured Drive output path.')

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/phase_boundary_outputs.zip'
    !cd /content && zip -qr phase_boundary_outputs.zip phase_boundary_outputs
    files.download(archive_path)
else:
    print(f'Outputs are in {saved_dir}')